# Multi-Model Benchmarking: No Context + MMR (k=15)

**Mrigi 23h** — Evaluates 14 models on two methods for Synthesis questions.

**Methods:** `no_context` (no retrieval), `mmr` at k=15  
**Dataset:** QuestionBank_GreenandNew_90.xlsx (Synthesis, 30 questions)  
**Setup:** Based on Mrigi 23g — cuda:0, local_files_only=True, float16  

**Models:**
- Baseline `meta-llama/Meta-Llama-3-8B-Instruct` (same baseline used in 23e/23f)
- DAPT `aleynabeste/AllClassesAbstracts70Mmodel_LR1e5` + COT FT
- `aleynabeste/model_LR1e5v3_synv2V2_step80` + COT FT
- `aleynabeste/model_LR1e5v3_synv2V2_final` + COT FT
- `aleynabeste/model_LR1e5v3_synv2_base_step80` + COT FT
- `aleynabeste/model_LR1e5v3_synv2_base_final` + COT FT
- `aleynabeste/model_LR1e5_fullpaper_longer_120M` + COT FT
- `aleynabeste/base_llama_intruct_COT_FT`

Each model is loaded, evaluated, then unloaded before the next to manage GPU memory.  
Checkpointing: results are saved to `results_23h_checkpoint.json` after every model — set `RESUME = True` in the loop cell to skip already-completed models if interrupted.

In [ ]:
import os
# # OOM Fix: limit max split size to reduce GPU memory fragmentation
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

import torch
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from collections import OrderedDict
import gc

from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load environment variables
with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            key, val = line[len('export '):].split('=', 1)
            os.environ[key] = val.strip('"').strip("'")

hf_token = os.environ.get("HF_TOKEN_BESTE")
if not hf_token:
    raise ValueError("HF_TOKEN_BESTE not found in env.sh")

os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print(f"\u2713 HuggingFace Hub token set (ending ...{hf_token[-4:]})")

In [ ]:
!nvidia-smi

In [ ]:
MODELS = OrderedDict([
    ('llama3_8b_instruct', {
        'name': 'meta-llama/Meta-Llama-3-8B-Instruct',
        'display_name': 'Llama-3-8B-Instruct',
        'gpu': 'cuda:0'
    }),
    ('dapt_lr1e5', {
        'name': 'aleynabeste/AllClassesAbstracts70Mmodel_LR1e5',
        'display_name': 'DAPT_LR1e5',
        'gpu': 'cuda:0'
    }),
    ('dapt_lr1e5_cot', {
        'name': 'aleynabeste/AllClassesAbstracts70Mmodel_LR1e5_COT',
        'display_name': 'DAPT_LR1e5_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_step80', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_step80',
        'display_name': 'synv2V2_step80',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_step80_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_step80_COT_FT',
        'display_name': 'synv2V2_step80_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_final', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_final',
        'display_name': 'synv2V2_final',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_final_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_final_COT_FT',
        'display_name': 'synv2V2_final_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_step80', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_step80',
        'display_name': 'synv2_base_step80',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_step80_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_step80_COT_FT',
        'display_name': 'synv2_base_step80_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_final', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_final',
        'display_name': 'synv2_base_final',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_final_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_final_COT_FT',
        'display_name': 'synv2_base_final_COT',
        'gpu': 'cuda:0'
    }),
    ('fullpaper_120M', {
        'name': 'aleynabeste/model_LR1e5_fullpaper_longer_120M',
        'display_name': 'fullpaper_120M_LR1e5',
        'gpu': 'cuda:0'
    }),
    ('fullpaper_120M_cot', {
        'name': 'aleynabeste/model_LR1e5_fullpaper_longer_120M_COT_FT',
        'display_name': 'fullpaper_120M_COT',
        'gpu': 'cuda:0'
    }),
    ('base_llama_cot', {
        'name': 'aleynabeste/base_llama_intruct_COT_FT',
        'display_name': 'base_llama_COT',
        'gpu': 'cuda:0'
    }),
])

K_MMR = 15


def load_model(model_key):
    config = MODELS[model_key]
    model_name = config['name']
    device = config['gpu']

    print(f"\n{'='*80}")
    print(f"Loading {config['display_name']} on {device}...")
    print(f"{'='*80}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_fast=True,
        token=hf_token,
        local_files_only=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=device,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        token=hf_token,
        local_files_only=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=100,
        temperature=0.1,
        do_sample=True,
    )
    llm = HuggingFacePipeline(pipeline=pipe)

    print(f"✓ {config['display_name']} loaded successfully on {device}")
    return llm, pipe, model, tokenizer


def cleanup_model(llm, pipe, model, tokenizer):
    del llm
    del pipe
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ GPU memory cleared")


print(f"✓ {len(MODELS)} models configured")
for k, v in MODELS.items():
    print(f"  {v['display_name']:30s}  {v['name']}")

In [ ]:
MC_PROMPT = """
You are answering multiple choice questions about zeolite synthesis. Select the most appropriate answer. You may use the provided context to inform your answer as well as your own knowledge.

Provide your answer in the following format EXACTLY:
Answer: [LETTER] - [One sentence explanation]

For example:
Answer: B - The synthesis temperature was increased to improve crystallization.

Choose only from options A, B, C, D, or E.
Context:
{context}

Question: {question}
"""

NO_CONTEXT_PROMPT = """You are answering multiple choice questions about zeolite synthesis. Select the most appropriate answer.

Provide your answer in the following format EXACTLY:
Answer: [LETTER] - [One sentence explanation]

For example:
Answer: B - The synthesis temperature was increased to improve crystallization.

Choose only from options A, B, C, D, or E.

Question: {question}"""


def extract_completion(full_response, tokenizer):
    assistant_tag = "<|start_header_id|>assistant<|end_header_id|>"
    if assistant_tag in full_response:
        completion = full_response.split(assistant_tag)[-1].strip()
    else:
        completion = full_response.strip()
    for token in ["<|eot_id|>", "<|end_of_text|>"]:
        completion = completion.replace(token, "").strip()
    return completion


def parse_mc_answer(completion):
    last_answer_pos = completion.rfind("Answer: ")
    if last_answer_pos != -1:
        model_answer = completion[last_answer_pos + 8].upper()
        if model_answer in ['A', 'B', 'C', 'D', 'E']:
            explanation = ""
            answer_end = completion[last_answer_pos:].split('\n')[0]
            if '-' in answer_end:
                explanation = answer_end.split('-', 1)[1].strip()
            return model_answer, explanation
    return None, None


def evaluate_mc_response(llm, query, correct_answer, context, tokenizer):
    """Evaluate a single MC question with context (used by MMR)."""
    prompt_text = MC_PROMPT.format(context=context, question=query)
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    try:
        full_response = llm(formatted_prompt)
        completion = extract_completion(full_response, tokenizer)
        model_answer, explanation = parse_mc_answer(completion)
        if model_answer is not None:
            return {
                'query': query, 'context': context,
                'model_answer': model_answer, 'correct_answer': correct_answer,
                'is_correct': model_answer == correct_answer,
                'full_response': completion, 'explanation': explanation or ""
            }
        return {
            'query': query, 'context': context,
            'model_answer': 'INVALID', 'correct_answer': correct_answer,
            'is_correct': False, 'full_response': completion,
            'error': 'No valid answer found'
        }
    except Exception as e:
        print(f"Processing error: {str(e)[:120]}")
        return {
            'query': query, 'context': context,
            'model_answer': 'ERROR', 'correct_answer': correct_answer,
            'is_correct': False, 'error': str(e)
        }


def evaluate_mc_no_context(llm, query, correct_answer, tokenizer):
    """Evaluate a single MC question without context."""
    prompt_text = NO_CONTEXT_PROMPT.format(question=query)
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    try:
        full_response = llm(formatted_prompt)
        completion = extract_completion(full_response, tokenizer)
        model_answer, explanation = parse_mc_answer(completion)
        if model_answer is not None:
            return {
                'query': query,
                'model_answer': model_answer, 'correct_answer': correct_answer,
                'is_correct': model_answer == correct_answer,
                'full_response': completion, 'explanation': explanation or ""
            }
        return {
            'query': query,
            'model_answer': 'INVALID', 'correct_answer': correct_answer,
            'is_correct': False, 'full_response': completion,
            'error': 'No valid answer found'
        }
    except Exception as e:
        print(f"Processing error: {str(e)[:120]}")
        return {
            'query': query,
            'model_answer': 'ERROR', 'correct_answer': correct_answer,
            'is_correct': False, 'error': str(e)
        }


def evaluate_method(llm, method_name, questions_df, tokenizer, k=15):
    """Run a single method over all questions. Clears GPU cache after each question."""
    results = []
    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df),
                       desc=f"Evaluating {method_name} (k={k})"):
        query = row['Question']
        correct_answer = row['Correct_Answer']
        try:
            if method_name == 'no_context':
                result = evaluate_mc_no_context(llm, query, correct_answer, tokenizer)
            elif method_name == 'mmr':
                context = get_mmr_context(query, k=k)
                result = evaluate_mc_response(llm, query, correct_answer, context, tokenizer)
            else:
                raise ValueError(f"Unknown method: {method_name}")
        except Exception as e:
            print(f"  \u26a0 Error on question: {str(e)[:100]}")
            result = {
                'query': query, 'model_answer': 'ERROR',
                'correct_answer': correct_answer, 'is_correct': False, 'error': str(e)
            }
        results.append(result)
        gc.collect()
        torch.cuda.empty_cache()

    correct_count = sum(1 for r in results if r.get('is_correct', False))
    total = len(results)
    accuracy = (correct_count / total * 100) if total > 0 else 0
    return {
        'method': method_name, 'k': k,
        'accuracy': accuracy, 'correct_count': correct_count,
        'total': total, 'detailed_results': results
    }


print("\u2713 Evaluation functions defined")

In [ ]:
from pathlib import Path

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

INDEX_DIRECTORY = Path("faiss_index")
INDEX_NAME = "index"

if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")

vector_db = FAISS.load_local(
    INDEX_DIRECTORY,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True
)
print(f"\u2713 Loaded FAISS index with {vector_db.index.ntotal} vectors")


def get_mmr_context(query, k=15):
    results = vector_db.max_marginal_relevance_search(query, k=k)
    return "\n\n".join([doc.page_content for doc in results])


print("\u2713 MMR retrieval function defined")

In [ ]:
QUESTION_FILE = 'QuestionBank_GreenandNew_90.xlsx'

df = pd.read_excel(QUESTION_FILE, sheet_name='Synthesis')
df = df.dropna(subset=['gpt_generated_question'])
df = df.rename(columns={
    'gpt_generated_question': 'Question',
    'gpt_generated_answer': 'Correct_Answer',
    'abstract': 'Context'
})
df['Correct_Answer'] = df['Correct_Answer'].str.strip().str.upper()
questions_df = df.reset_index(drop=True)

print(f"\u2713 Loaded {len(questions_df)} Synthesis questions")

In [ ]:
# Set RESUME=True to load existing checkpoint and skip already-completed models.
# Set RESUME=False for a fresh run (checkpoint will be overwritten).
RESUME = False
CHECKPOINT_PATH = 'results_23h_checkpoint.json'

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_PATH = f'results_23h_complete_{timestamp}.json'


def _is_model_complete(res):
    return (
        isinstance(res, dict)
        and 'no_context' in res and 'mmr' in res
        and res['no_context'].get('total', 0) > 0
        and res['mmr'].get('total', 0) > 0
    )


if RESUME and os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        all_results = json.load(f)
    done = [k for k, v in all_results.items() if _is_model_complete(v)]
    print(f"✓ Resuming from {CHECKPOINT_PATH} — {len(done)} models already complete:")
    for d in done:
        print(f"    - {d}")
else:
    all_results = {}
    if RESUME:
        print(f"No checkpoint found at {CHECKPOINT_PATH} — starting fresh.")
    else:
        print(f"Fresh run — checkpoint {CHECKPOINT_PATH} will be overwritten.")

for model_key, model_config in MODELS.items():
    display_name = model_config['display_name']

    if _is_model_complete(all_results.get(display_name)):
        print(f"\n[Skip] {display_name} — already complete in checkpoint")
        continue

    print(f"\n{'#'*80}")
    print(f"MODEL: {display_name}")
    print(f"{'#'*80}")

    # Load
    try:
        llm, pipe, model, tokenizer = load_model(model_key)
    except Exception as e:
        print(f"✗ Failed to load {display_name}: {e}")
        all_results[display_name] = {'load_error': str(e)}
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump(all_results, f, indent=2, default=str)
        continue

    if display_name not in all_results or 'load_error' in all_results[display_name]:
        all_results[display_name] = {}

    # no_context
    print(f"\n--- no_context ---")
    try:
        result = evaluate_method(llm, 'no_context', questions_df, tokenizer, k=K_MMR)
        all_results[display_name]['no_context'] = result
        print(f"✓ no_context: {result['accuracy']:.2f}% ({result['correct_count']}/{result['total']})")
    except Exception as e:
        print(f"✗ no_context error: {e}")
        all_results[display_name]['no_context'] = {'error': str(e), 'accuracy': 0, 'correct_count': 0, 'total': 0}

    # mmr k=15
    print(f"\n--- mmr (k={K_MMR}) ---")
    try:
        result = evaluate_method(llm, 'mmr', questions_df, tokenizer, k=K_MMR)
        all_results[display_name]['mmr'] = result
        print(f"✓ mmr k={K_MMR}: {result['accuracy']:.2f}% ({result['correct_count']}/{result['total']})")
    except Exception as e:
        print(f"✗ mmr error: {e}")
        all_results[display_name]['mmr'] = {'error': str(e), 'accuracy': 0, 'correct_count': 0, 'total': 0}

    # Cleanup
    cleanup_model(llm, pipe, model, tokenizer)

    # Save after each model (checkpoint + timestamped snapshot)
    for path in (CHECKPOINT_PATH, SAVE_PATH):
        with open(path, 'w') as f:
            json.dump(all_results, f, indent=2, default=str)
    sz = os.path.getsize(SAVE_PATH) / 1024
    print(f"✓ Saved: {SAVE_PATH} ({sz:.0f} KB) + checkpoint {CHECKPOINT_PATH}")

print(f"\n{'='*80}")
print("All models evaluated.")
print(f"{'='*80}")

In [ ]:
print("\n" + "="*90)
print("RESULTS SUMMARY: No Context vs MMR (k=15) — Synthesis (n=30)")
print("="*90)
print(f"{'Model':<35} {'no_context':>12} {'mmr_k15':>12} {'delta':>10}")
print("-"*90)

for display_name, res in all_results.items():
    if 'load_error' in res:
        print(f"{display_name:<35} {'LOAD ERROR':>12}")
        continue
    nc_acc = res.get('no_context', {}).get('accuracy', float('nan'))
    mmr_acc = res.get('mmr', {}).get('accuracy', float('nan'))
    try:
        delta = mmr_acc - nc_acc
        sign = '+' if delta >= 0 else ''
        print(f"{display_name:<35} {nc_acc:>10.2f}%  {mmr_acc:>10.2f}%  {sign}{delta:>+.2f}%")
    except Exception:
        print(f"{display_name:<35} {'ERR':>12} {'ERR':>12}")

print("="*90)

In [ ]:
model_names = []
nc_accs = []
mmr_accs = []

for display_name, res in all_results.items():
    if 'load_error' in res:
        continue
    model_names.append(display_name)
    nc_accs.append(res.get('no_context', {}).get('accuracy', 0))
    mmr_accs.append(res.get('mmr', {}).get('accuracy', 0))

x = np.arange(len(model_names))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(max(14, len(model_names) * 1.2), 7))
bars1 = ax.bar(x - bar_width/2, nc_accs, bar_width, label='no_context',
               color='#FF9800', alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + bar_width/2, mmr_accs, bar_width, label=f'mmr (k={K_MMR})',
               color='#9C27B0', alpha=0.85, edgecolor='black')

for bar, acc in zip(bars1, nc_accs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')
for bar, acc in zip(bars2, mmr_accs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('23h: No Context vs MMR (k=15) — Synthesis (n=30)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, 105)

plt.tight_layout()
chart_path = f'results_23h_chart_{timestamp}.svg'
fig.savefig(chart_path, format='svg', bbox_inches='tight')
plt.show()
print(f"\u2713 Chart saved: {chart_path}")

In [ ]:
rows = []
for display_name, res in all_results.items():
    if 'load_error' in res:
        continue
    for method in ['no_context', 'mmr']:
        r = res.get(method, {})
        rows.append({
            'model': display_name,
            'method': method,
            'k': K_MMR if method == 'mmr' else 'n/a',
            'accuracy': r.get('accuracy', 0),
            'correct': r.get('correct_count', 0),
            'total': r.get('total', 0)
        })

csv_df = pd.DataFrame(rows)
csv_path = f'results_23h_{timestamp}.csv'
csv_df.to_csv(csv_path, index=False)
print(f"\u2713 CSV saved: {csv_path}")
print(csv_df.to_string(index=False))

In [ ]:
final_path = f'results_23h_complete_{timestamp}.json'
with open(final_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\u2713 Final results saved: {final_path} ({size_mb:.1f} MB)")

# Verify
total_saved = sum(
    len(res.get(m, {}).get('detailed_results', []))
    for res in all_results.values()
    for m in ['no_context', 'mmr']
    if 'load_error' not in res
)
print(f"  Total question-level records: {total_saved}")